In [1]:
import mysql.connector
import pandas as pd

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Mansi101",
    database="sales_management"
)

print("Connected successfully!")

Connected successfully!


RETRIEVE SALES DATA:

In [2]:
query = """
SELECT
    o.OrderID,
    c.CustomerName,
    c.City,
    p.ProductName,
    p.Category,
    o.OrderDate,
    o.Quantity,
    p.Price,
    o.Quantity * p.Price AS Revenue
FROM Orders o
JOIN Customers c
ON o.CustomerID = c.CustomerID
JOIN Products p
ON o.ProductID = p.ProductID
"""

sales = pd.read_sql(query, connection)

print(sales)

   OrderID  CustomerName       City   ProductName     Category   OrderDate  \
0     1001  Rahul Sharma      Delhi        Laptop  Electronics  2026-01-05   
1     1006  Rahul Sharma      Delhi          Desk    Furniture  2026-02-05   
2     1002   Priya Singh     Mumbai         Mouse  Electronics  2026-01-07   
3     1007   Priya Singh     Mumbai        Laptop  Electronics  2026-02-10   
4     1003    Amit Kumar  Bangalore  Office Chair    Furniture  2026-01-10   
5     1008    Amit Kumar  Bangalore         Mouse  Electronics  2026-02-15   
6     1004    Neha Gupta      Delhi      Keyboard  Electronics  2026-01-15   
7     1005   Rohan Mehta       Pune      Notebook   Stationery  2026-01-20   

   Quantity    Price  Revenue  
0         1  55000.0  55000.0  
1         1  12000.0  12000.0  
2         3    800.0   2400.0  
3         1  55000.0  55000.0  
4         2   7500.0  15000.0  
5         5    800.0   4000.0  
6         2   1500.0   3000.0  
7        10    100.0   1000.0  


C:\Users\ACER.DESKTOP-2919HSJ\AppData\Local\Temp\ipykernel_11544\2702873369.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales = pd.read_sql(query, connection)


CALCULATE SALES KPI's:

In [3]:
total_revenue = sales["Revenue"].sum()

total_orders = sales["OrderID"].nunique()

total_quantity = sales["Quantity"].sum()

average_order_value = total_revenue / total_orders

print("Total Revenue:", total_revenue)
print("Total Orders:", total_orders)
print("Total Quantity:", total_quantity)
print("Average Order Value:", average_order_value)

Total Revenue: 147400.0
Total Orders: 8
Total Quantity: 25
Average Order Value: 18425.0


TOP 5 PRODUCTS:


In [4]:
top_products = (
    sales.groupby("ProductName")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

print(top_products)

    ProductName   Revenue
0        Laptop  110000.0
1  Office Chair   15000.0
2          Desk   12000.0
3         Mouse    6400.0
4      Keyboard    3000.0


CATEGORY ANALYSIS:

In [5]:
category_sales = (
    sales.groupby("Category")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

print(category_sales)

      Category   Revenue  Quantity
0  Electronics  119400.0        12
1    Furniture   27000.0         3
2   Stationery    1000.0        10


CITY ANALYSIS:

In [6]:
city_sales = (
    sales.groupby("City")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print(city_sales)

        City  Revenue
0      Delhi  70000.0
1     Mumbai  57400.0
2  Bangalore  19000.0
3       Pune   1000.0


MONTHLY SALES ANALYSIS:

In [7]:
sales["OrderDate"] = pd.to_datetime(sales["OrderDate"])

monthly_sales = (
    sales.groupby(
        sales["OrderDate"].dt.to_period("M")
    )["Revenue"]
    .sum()
    .reset_index()
)

monthly_sales["OrderDate"] = (
    monthly_sales["OrderDate"].astype(str)
)

print(monthly_sales)

  OrderDate  Revenue
0   2026-01  76400.0
1   2026-02  71000.0


EXPORT DATA TO EXCEL:

In [8]:
with pd.ExcelWriter(
    "Sales_Management_Report.xlsx",
    engine="openpyxl"
) as writer:

    sales.to_excel(
        writer,
        sheet_name="Sales Data",
        index=False
    )

    category_sales.to_excel(
        writer,
        sheet_name="Category Sales",
        index=False
    )

    city_sales.to_excel(
        writer,
        sheet_name="City Sales",
        index=False
    )

    top_products.to_excel(
        writer,
        sheet_name="Top Products",
        index=False
    )

    monthly_sales.to_excel(
        writer,
        sheet_name="Monthly Sales",
        index=False
    )

print("Excel report created successfully!")

Excel report created successfully!
